#Extração e processamento dos dados brutos para Cloud Analysis

Este notebook documenta o fluxo de aquisição e organização inicial dos dados brutos referentes às amostras PC (Controle) e PM (Mutada). O processo contempla o download dos arquivos a partir de repositório em nuvem, a extração seletiva de diretórios e o mapeamento sistemático dos identificadores de sequenciamento. Esta etapa é fundamental para assegurar a integridade e a rastreabilidade dos arquivos que serão submetidos ao processamento secundário, servindo de base para a futura geração das matrizes de expressão espacial.

In [ ]:
import os
import pandas as pd
from google.colab import drive

#Paciente


In [ ]:
%cd "/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/raw/patient"

#extração dos arquivos compactados
!unzip -q swisstransfer_d6d89591-9e72-48d1-a3d2-e51643bb738c.zip

O arquivo compactado foi movido para outra pasta.

Verificando a integridade dos arquivos descompactados:

In [ ]:
#novo caminho para o arquivo compactado
zip_folder = "/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/raw/zip/zip_patient/"

#caminho da pasta com os arquivos descompactados
target_dir = "/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/raw/patient/"

print("VERIFICAÇÃO DE INTEGRIDADE\n")

#localização do zip
zips = [f for f in os.listdir(zip_folder) if f.endswith('.zip')]
if zips:
    zip_path = os.path.join(zip_folder, zips[0])
    print(f"Arquivo original: {zips[0]}")
    !unzip -l "{zip_path}" | grep -E "FASTQ|TIF|fastq.gz"
else:
    print("arquivo zip não encontrado")

#verificação da pasta de destino
print("\nArquivos extraídos")
!ls -lh "{target_dir}" | grep -E "FASTQ|TIF|fastq.gz"

Dado que o dataset exportado apresenta o mesmo tamanho que o do zip disponibilizado, constatou-se sua integridade. Prossegiu-se então para a importação para o Cloud Analysis.

##Envio dos dados para o Cloud Analysis


In [ ]:
%cd "/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/raw/patient"

# download the 10x Genomics Cloud CLI package using their official curl command.
!curl -f -o txg-linux-v4.0.0.tar.gz "https://cf.10xgenomics.com/cloud-cli/v4.0.0/txg-linux-v4.0.0.tar.gz"

# unzip the tool
!tar -zxvf txg-linux-v4.0.0.tar.gz

# start upload
!./txg-linux-v4.0.0/txg files upload --project-id qSl903k905T1ufyZbg5wl4zA --access-token 027510fa645c19ea91622f6b7253f8041bcfaffc7136e554bee66d2ac4828433 .

In [ ]:
# Função para classificar os Outputs (Tabela 2)
def classify_output(file):
    if "binned_outputs" in file: return "Matrizes agregadas (Bins)"
    if "segmented_outputs" in file: return "Dados segmentados"
    if "spatial_outputs" in file: return "Metadados posicionais"
    if "barcode_mappings" in file: return "Mapeamento de identificadores"
    if "loupe" in file: return "Visualização interativa"
    return None

#Controle

In [ ]:
%cd "/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/raw/control"

#extração dos dados
!unzip -q swisstransfer_e60532ac-318e-45db-a06b-fbf3d8b79690.zip

Os dados foram sequenciados novamente e a pasta com a segunda rodada estava dentro da primeira pasta. Assim, foi necessária uma segunda extração:

In [ ]:
%cd "/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/raw/control"

#extração dos dados de resequencing
!unzip -q swisstransfer_e60532ac-318e-45db-a06b-fbf3d8b79690.zip VisiumHD_ReSequencing-NormalControl.tar

Após o envio dos dados para o Cloud Analysis, o seguinte erro foi encontrado:

TXRNGR10011: IO error in FASTQ file: '7588d947-4070-4b1f-8790-a646a35284e0/573d5777-d0b5-4595-8f2e-3d079015d8f5/CTRL31_VISHD-AK40380-AK40381_22VVGKLT4_S1_L003_R2_001.fastq.gz', line: 289015600: unexpected end of file

Compreendeu-se que a extração do arquivo L003_R2 não foi bem sucedida, e ela foi feita novamente:

In [ ]:
#ver quais arquivos tem dentro do zip que foi disponibilizado
!unzip -l "/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/raw/control/swisstransfer_e60532ac-318e-45db-a06b-fbf3d8b79690.zip"

In [ ]:
#extraindo o .tar para pasta local (/content/)
#tira o .tar de dentro do .zip e salva na pasta temporária do Colab
!unzip -j "/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/raw/control/swisstransfer_e60532ac-318e-45db-a06b-fbf3d8b79690.zip" "VisiumHD_ReSequencing-NormalControl.tar" -d /content/

In [ ]:
#verificação do conteúdo
!tar -tf /content/VisiumHD_ReSequencing-NormalControl.tar!tar -tf /content/VisiumHD_ReSequencing-NormalControl.tar

In [ ]:
#apagando o arquivo corrompido antes de substituir
!rm "/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/raw/control/CTRL31_VISHD-AK40380-AK40381_22VVGKLT4_S1_L003_R2_001.fastq.gz"

In [ ]:
#extraindo apenas o arquivo para substituir o corrompido, com o caminho completo e enviando direto para a pasta de destino
!tar -xvf /content/VisiumHD_ReSequencing-NormalControl.tar \
--strip-components=3 \
-C "/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/raw/control/" \
"X208SC25042012-Z01-F002/01.RawData/CTRL31_VISHD/CTRL31_VISHD-AK40380-AK40381_22VVGKLT4_S1_L003_R2_001.fastq.gz"

In [ ]:
%%time

#definição dos caminhos
zip_path = "/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/raw/zip/zip_control/swisstransfer_e60532ac-318e-45db-a06b-fbf3d8b79690.zip"
target_dir = "/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/raw/control/"

#extração com o- (overwrite) para substituir os arquivos com problema
!unzip -o -j "{zip_path}" \
"CTRL31_VISHD-AK40380-AK40381_22TJWNLT4_S1_L004_R1_001.fastq.gz" \
"CTRL31_VISHD-AK40380-AK40381_22TJWNLT4_S1_L004_I1_001.fastq.gz" \
"CTRL31_VISHD-AK40380-AK40381_22TJWNLT4_S1_L004_I2_001.fastq.gz" \
-d "{target_dir}"

#verificação de segurança
print("\n VERIFICAÇÃO DOS TAMANHOS ")
!ls -lh "{target_dir}" | grep "L004"

In [ ]:
%%time

#caminhos
zip_path = "/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/raw/zip/zip_control/swisstransfer_e60532ac-318e-45db-a06b-fbf3d8b79690.zip"
target_dir = "/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/raw/control/"

#remoção do arquivo corrompido
!rm "{target_dir}CTRL31_VISHD-AK40380-AK40381_22TJWNLT4_S1_L004_R1_001.fastq.gz"

#extração do arquivo R1 novamente
!unzip -j "{zip_path}" "CTRL31_VISHD-AK40380-AK40381_22TJWNLT4_S1_L004_R1_001.fastq.gz" -d "{target_dir}"

#verificação
print("\n VERIFICAÇÃO ")
!ls -lh "{target_dir}" | grep "L004_R1"

In [ ]:
import os

#caminhos
zip_path = "/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/raw/zip/zip_control/swisstransfer_e60532ac-318e-45db-a06b-fbf3d8b79690.zip"
target_dir = "/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/raw/control/"
tar_path = "/content/VisiumHD_ReSequencing-NormalControl.tar"

print(" RELATÓRIO DE INTEGRIDADE \n")

#extração do L004 e imagens do zip disponibilizado
!unzip -l "{zip_path}" | grep -E "L004|TIF"

#extração do L003 do TAR no content local
if os.path.exists(tar_path):
    print("\n verificando arquivo TAR")
    !tar -tvf "{tar_path}" | grep "L003"
else:
    print("\n arquivo TAR não encontrado")

#arquivos do Drive
print("\n VERIFICAÇÃO DE CONTEÚDO DA PASTA NO DRIVE")
!ls -lh "{target_dir}" | grep -E "L003|L004|TIF"

##Verificação de integridade dos dados


In [ ]:
#verificar espaço no disco local
!du -sh /content/*

In [ ]:
#deletando o TAR para liberar espaço local
!rm /content/VisiumHD_ReSequencing-NormalControl.tar

Para que não ocorra problemas na importação dos dados para o Cloud Analysis, eles foram copiados para uma pasta no disco local do Colab:

In [ ]:
%%time
import os

#criar pasta local
!mkdir -p /content/upload_control

#arquivos para copiar do Drive para a pasta local
drive_path = "/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/raw/control/"
local_path = "/content/upload_control/"

#cp com flag v para acompanhar progresso dos arquivos
!cp -v {drive_path}* {local_path}

#verificar espaço
!du -sh /content/upload_control

O tempo de 31 minutos para copiar 51GB confirma que a conexão entre o Drive e o disco local do Colab estava lenta, o que provavelmente era o que causava as quedas no upload direto.

Verificação de integridade dos dados:

In [ ]:
import os

#caminhos
zip_path = "/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/raw/zip/zip_control/swisstransfer_e60532ac-318e-45db-a06b-fbf3d8b79690.zip"
local_path = "/content/upload_control/"

print(" COMPARAÇÃO PASTA LOCAL VS ZIP \n")

#tamanhos dos arquivos no zip original L004
print("TAMANHOS ORIGINAIS L004")
!unzip -l "{zip_path}" | grep -E "L004|TIF"

#tamanhos dos arquivos no zip original L003 resequencing sem extrair
print("TAMANHOS ORIGINAIS L003")
!unzip -p "{zip_path}" "VisiumHD_ReSequencing-NormalControl.tar" | tar -tvf - | grep "L003"

#tamanhos dos arquivos na pasta local
print("\nTAMANHOS LOCAIS")
!ls -lh "{local_path}" | grep -E "L003|L004|TIF"

In [ ]:
Houve divergência no tamanho do L004_R1, com 4.5G, mas deveria ter 6.4G. Foi feita a extração novamente:

In [ ]:
#verificar conteúdo da pasta local
!ls -lh /content/

#remover arquivo corrompido de 4.5G da pasta local
!rm "/content/upload_control/CTRL31_VISHD-AK40380-AK40381_22TJWNLT4_S1_L004_R1_001.fastq.gz"

#remover lixeira e dados temporários do disco local para liberar espaço
!rm -rf /content/sample_data

In [ ]:
%%time
zip_path = "/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/raw/zip/zip_control/swisstransfer_e60532ac-318e-45db-a06b-fbf3d8b79690.zip"
local_dir = "/content/upload_control/"

# Extraindo o L004 R1 novamente
!unzip -j "{zip_path}" "CTRL31_VISHD-AK40380-AK40381_22TJWNLT4_S1_L004_R1_001.fastq.gz" -d "{local_dir}"

print("\nCONFERÊNCIA DE TAMANHO")
!ls -lh "{local_dir}" | grep "L004_R1"

Verificação final de integridade dos arquivos:

In [ ]:
#caminho
local_path = "/content/upload_control/"

#tamanhos dos arquivos na pasta local
print("\nTAMANHOS LOCAIS:")
!ls -lh "{local_path}" | grep -E "L003|L004|TIF"

Verificou-se então que os dados finalmente estão íntegros e foi feita a importação para o Cloud Analysis.

##Importação dos dados para o Cloud Analysis

In [ ]:
%%time
#caminho
%cd /content/upload_control

#ferramenta de upload disponível no projeto
!tar -zxvf txg-linux-v4.0.0.tar.gz --strip-components=1

#upload
!./txg files upload --project-id H3zHo3wVSSO0woFvM7JHAQ --access-token 027510fa645c19ea91622f6b7253f8041bcfaffc7136e554bee66d2ac4828433 .

#Exportando tabelas descritivas dos dados

In [ ]:
drive.mount('/content/drive')

#função para tamanhos de arquivos
def obter_tamanho_formatado(tamanho_em_bytes):
    if tamanho_em_bytes == 0: return "0 B"
    unidades = ("B", "KB", "MB", "GB")
    i = 0
    while tamanho_em_bytes >= 1024 and i < len(unidades) - 1:
        tamanho_em_bytes /= 1024
        i += 1
    return f"{tamanho_em_bytes:.2f} {unidades[i]}"

#de para tipo de arquivo
def classificar_entrada(nome_arquivo):
    tipo = "Outros"
    leitura = "-"
    if "_R1_" in nome_arquivo: tipo = "R1"
    elif "_R2_" in nome_arquivo: tipo = "R2"
    elif "_I1_" in nome_arquivo: tipo = "I1"
    elif "_I2_" in nome_arquivo: tipo = "I2"
    #de para lane/leitura
    for l in ["L003", "L004", "L007", "L008"]:
        if l in nome_arquivo:
            leitura = l.replace("L00", "Lane ")
    #de para imagens
    if nome_arquivo.lower().endswith('.tif'):
        if any(x in nome_arquivo.lower() for x in ["cytassist", "control.tif", "myrf.tif"]):
            tipo = "Imagem CytAssist"
        else:
            tipo = "Imagem de microscopia"

    return tipo, leitura

#de para outputs
def classificar_saida(nome_arquivo):
    if "binned_outputs" in nome_arquivo: return "Matrizes agregadas (Bins)"
    if "segmented_outputs" in nome_arquivo: return "Dados segmentados"
    if "spatial_outputs" in nome_arquivo: return "Metadados posicionais"
    if "barcode_mappings" in nome_arquivo: return "Mapeamento de identificadores"
    if "loupe" in nome_arquivo: return "Visualização interativa"
    return None

#caminho para salvar as tabelas
caminho_tabelas = "/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/tables"

##Inputs

In [ ]:
#inputs PM
caminho_pm = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/raw/patient/'
arquivos_pm = [f for f in os.listdir(caminho_pm) if os.path.isfile(os.path.join(caminho_pm, f)) and "txg" not in f]

dados_pm_entrada = []
for f in arquivos_pm:
    tipo, lane = classificar_entrada(f)
    tamanho = obter_tamanho_formatado(os.path.getsize(os.path.join(caminho_pm, f)))
    dados_pm_entrada.append({
        'Paciente': 'PM',
        'Nome do arquivo': f,
        'Tipo de arquivo': tipo,
        'Leitura': lane,
        'Formato': os.path.splitext(f)[1],
        'Tamanho': tamanho
    })

df_pm_entrada = pd.DataFrame(dados_pm_entrada).sort_values(by=['Leitura', 'Tipo de arquivo'])
display(df_pm_entrada)


#salva no Drive
df_pm_entrada.to_csv(f"{caminho_tabelas}/Tabela 1. Descrição do conjunto de dados brutos de estudo de caso_PM.csv", sep=';', index=False)

In [ ]:
#inputs PC
caminho_pc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/raw/control/'
arquivos_pc = [f for f in os.listdir(caminho_pc) if os.path.isfile(os.path.join(caminho_pc, f)) and "txg" not in f]

dados_pc_entrada = []
for f in arquivos_pc:
    tipo, lane = classificar_entrada(f)
    tamanho = obter_tamanho_formatado(os.path.getsize(os.path.join(caminho_pc, f)))
    dados_pc_entrada.append({
        'Paciente': 'PC',
        'Nome do arquivo': f,
        'Tipo de arquivo': tipo,
        'Leitura': lane,
        'Formato': os.path.splitext(f)[1],
        'Tamanho': tamanho
    })

df_pc_entrada = pd.DataFrame(dados_pc_entrada).sort_values(by=['Leitura', 'Tipo de arquivo'])

display(df_pc_entrada)

#salva no Drive
df_pc_entrada.to_csv(f"{caminho_tabelas}/Tabela 1. Descrição do conjunto de dados brutos de estudo de caso_PC.csv", sep=';', index=False)

##Outputs

In [ ]:
#outputs PM
caminho_pm_saida = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/processed/patient/'
arquivos_pm_saida = [f for f in os.listdir(caminho_pm_saida) if os.path.isfile(os.path.join(caminho_pm_saida, f))]

dados_pm_saida = []
for f in arquivos_pm_saida:
    tipo = classificar_saida(f)
    if tipo:
        tamanho = obter_tamanho_formatado(os.path.getsize(os.path.join(caminho_pm_saida, f)))
        extensao = '.tar.gz' if f.endswith('.tar.gz') else os.path.splitext(f)[1]
        dados_pm_saida.append({
            'Paciente': 'PM',
            'Arquivo (Tipo)': tipo,
            'Formato': extensao,
            'Tamanho': tamanho
        })

df_pm_saida = pd.DataFrame(dados_pm_saida).sort_values(by='Arquivo (Tipo)')

display(df_pm_saida)

#salva no Drive
df_pm_saida.to_csv(f"{caminho_tabelas}/Tabela 2. Descrição do conjunto de dados de estudo de caso_PM.csv", sep=';', index=False, encoding='utf-8-sig')

In [ ]:
#outputs PC
caminho_pc_saida = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/processed/control/'
arquivos_pc_saida = [f for f in os.listdir(caminho_pc_saida) if os.path.isfile(os.path.join(caminho_pc_saida, f))]

dados_pc_saida = []
for f in arquivos_pc_saida:
    tipo = classificar_saida(f)
    if tipo:
        tamanho = obter_tamanho_formatado(os.path.getsize(os.path.join(caminho_pc_saida, f)))
        extensao = '.tar.gz' if f.endswith('.tar.gz') else os.path.splitext(f)[1]
        dados_pc_saida.append({
            'Paciente': 'PC',
            'Arquivo (Tipo)': tipo,
            'Formato': extensao,
            'Tamanho': tamanho
        })

df_pc_saida = pd.DataFrame(dados_pc_saida).sort_values(by='Arquivo (Tipo)')

display(df_pc_saida)

#salva no Drive
df_pc_saida.to_csv(f"{caminho_tabelas}/Tabela 2. Descrição do conjunto de dados de estudo de caso_PC.csv", sep=';', index=False, encoding='utf-8-sig')